
# Governance 05 · Row filter by region and column mask on claim free text (Unity Catalog)

The Lakeflow pipeline attaches these policies to the claims tables it publishes
(`silver_claims`, `silver_claims_ai_checked`, see `src/pipeline/lrc_ingest_pipeline.sql`):

| Policy | Function | Effect |
|---|---|---|
| **Row filter** on `region` | `lrc_region_filter(region)` | A claims team sees only its own book: membership of a group listed in `lrc_region_access` for that region. Workspace admins (the pipeline owner) see all regions. |
| **Column mask** on `loss_description` | `lrc_mask_free_text(s)` | GDPR: only members of `lrc_pii_readers` (and admins) see the raw text; everyone else sees it with emails, phone numbers and contact names redacted by `lrc_redact`. |

Gen AI (`ai_mask`) finds personal data the regexes miss, such as names. The column mask *enforces* redaction for
every query, on every surface (SQL, Genie, dashboards, Power BI DirectQuery), regardless of who wrote the query.

This notebook runs before the pipeline in the `lrc_end_to_end` job, because the pipeline references the functions.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "loss_ratio_coworker")
CATALOG = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema")
FQ = f"`{CATALOG}`.`{SCHEMA}`"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")

## 1 · Region access mapping (which group may see which book)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.lrc_region_access (group_name STRING, region STRING)
COMMENT 'Row-filter mapping for claims tables: members of group_name may see claims in region.'
""")
spark.sql(f"""
INSERT INTO {FQ}.lrc_region_access VALUES
  ('lrc_claims_uk', 'UK & Ireland'), ('lrc_claims_france', 'France'), ('lrc_claims_dach', 'DACH'),
  ('lrc_claims_southern_europe', 'Southern Europe'), ('lrc_claims_benelux', 'Benelux'),
  ('lrc_claims_all', 'UK & Ireland'), ('lrc_claims_all', 'France'), ('lrc_claims_all', 'DACH'),
  ('lrc_claims_all', 'Southern Europe'), ('lrc_claims_all', 'Benelux')
""")
display(spark.table(f"{FQ}.lrc_region_access"))

| group_name | region |
|---|---|
| lrc_claims_uk | UK & Ireland |
| lrc_claims_france | France |
| lrc_claims_dach | DACH |
| lrc_claims_southern_europe | Southern Europe |
| lrc_claims_benelux | Benelux |
| lrc_claims_all | UK & Ireland |
| lrc_claims_all | France |
| lrc_claims_all | DACH |
| lrc_claims_all | Southern Europe |
| lrc_claims_all | Benelux |

## 2 · Policy functions

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.lrc_region_filter(region STRING)
RETURNS BOOLEAN
COMMENT 'Row filter for claims tables: admins see all regions; others only regions mapped to a group they belong to.'
RETURN is_member('admins')
    OR EXISTS (SELECT 1 FROM {FQ}.lrc_region_access a
               WHERE a.region = lrc_region_filter.region AND is_account_group_member(a.group_name))
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.lrc_redact(s STRING)
RETURNS STRING
COMMENT 'Redacts email addresses, phone numbers and contact names from free text.'
RETURN regexp_replace(regexp_replace(regexp_replace(s,
         '[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\\\.[A-Za-z]{{2,}}', '[email]'),
         '\\\\+?[0-9][0-9 ]{{7,}}[0-9]', '[phone]'),
         '([Cc]ontact|[Cc]all|[Aa]sk for|[Ss]peak to) [A-Z][a-z]+( [A-Z][A-Za-z\\']+)?', '$1 [name]')
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.lrc_mask_free_text(s STRING)
RETURNS STRING
COMMENT 'Column mask for claim free text: raw text for admins and lrc_pii_readers, redacted for everyone else.'
RETURN CASE WHEN is_member('admins') OR is_account_group_member('lrc_pii_readers') THEN s ELSE {FQ}.lrc_redact(s) END
""")
print("policy functions ready")

policy functions ready


## 3 · What the policies do (evidence)
Left: the raw text an authorised reader sees. Right: what any other analyst, Genie user or Power BI report sees.

In [ ]:
if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.silver_claims"):
    display(spark.sql(f"""
      SELECT claim_id, region, loss_description AS seen_by_pii_readers, {FQ}.lrc_redact(loss_description) AS seen_by_everyone_else
      FROM {FQ}.silver_claims WHERE loss_description RLIKE '@|\\\\+44' LIMIT 8"""))
    display(spark.sql(f"""
      SELECT a.group_name, COUNT(*) AS claims_visible
      FROM {FQ}.silver_claims c JOIN {FQ}.lrc_region_access a ON a.region = c.region
      GROUP BY a.group_name ORDER BY a.group_name"""))

| claim_id | region | seen_by_pii_readers | seen_by_everyone_else |
|---|---|---|---|
| CLM-3073814 | UK & Ireland | Burst pipe on the third floor caused water damage to offices below in Dublin. Contact Anna Becker on +44 7700 549807 or anna@example.com | Burst pipe on the third floor caused water damage to offices below in Dublin. Contact [name] on [phone] or [email] |
| CLM-3073822 | Benelux | Heavy seas caused containers to shift and fall overboard en route to Antwerp. Contact Sophie Martin on +44 7700 839104 or sophie@example.com | Heavy seas caused containers to shift and fall overboard en route to Antwerp. Contact [name] on [phone] or [email] |
| CLM-3073903 | Benelux | Delivery van rear-ended at traffic lights in Rotterdam, bumper and tailgate damaged. Contact Sophie Martin on +44 7700 253185 or sophie@example.com | Delivery van rear-ended at traffic lights in Rotterdam, bumper and tailgate damaged. Contact [name] on [phone] or [email] |
| CLM-3073906 | UK & Ireland | Company car collided with a parked vehicle while reversing in Dublin. Contact Carlos Ruiz on +44 7700 127259 or carlos@example.com | Company car collided with a parked vehicle while reversing in Dublin. Contact [name] on [phone] or [email] |
| CLM-3073908 | UK & Ireland | Container seal broken, part of the cargo missing on arrival in Dublin. Contact Luca Rossi on +44 7700 510172 or luca@example.com | Container seal broken, part of the cargo missing on arrival in Dublin. Contact [name] on [phone] or [email] |
| CLM-3073912 | Southern Europe | River flooding entered the ground floor of the Milan depot, water 40cm deep. Contact Carlos Ruiz on +44 7700 379512 or carlos@example.com | River flooding entered the ground floor of the Milan depot, water 40cm deep. Contact [name] on [phone] or [email] |
| CLM-3073963 | Southern Europe | Pedestrian injured when our delivery vehicle mounted the kerb in Porto. Contact Carlos Ruiz on +44 7700 408047 or carlos@example.com | Pedestrian injured when our delivery vehicle mounted the kerb in Porto. Contact [name] on [phone] or [email] |
| CLM-3073978 | Southern Europe | Pallets crushed during loading, consignment for Milan rejected. Contact James O'Brien on +44 7700 859887 or james@example.com | Pallets crushed during loading, consignment for Milan rejected. Contact [name] on [phone] or [email] |

| group_name | claims_visible |
|---|---|
| lrc_claims_all | 3716 |
| lrc_claims_benelux | 400 |
| lrc_claims_dach | 1020 |
| lrc_claims_france | 679 |
| lrc_claims_southern_europe | 617 |
| lrc_claims_uk | 1000 |

## 4 · Policies attached in Unity Catalog

In [ ]:
for t in ["silver_claims", "silver_claims_ai_checked"]:
    if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{t}"):
        rows = [r for r in spark.sql(f"DESCRIBE TABLE EXTENDED {FQ}.{t}").collect()
                if any(k in (r[0] or "") for k in ("Row Filter", "Column Mask", "Masks", "Filter"))]
        print(t, [(r[0], r[1]) for r in rows] or "(policies are attached when the pipeline next refreshes)")

silver_claims [('Row Filter', '`<catalog>`.`loss_ratio_coworker`.`lrc_region_filter` ON (region)'), ('# Column Masks', '')]
silver_claims_ai_checked (policies are attached when the pipeline next refreshes)
